# Project19 Database Explorer

Use this notebook to query `annotations.db`, inspect annotated frames, and plot Alpamayo prediction paths.

It is read-only by default. The notebook connects to SQLite directly and assumes it is being run from the project root or from inside `llm-model-tests/`.

In [ ]:
from pathlib import Path
import io
import json
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

cwd = Path.cwd().resolve()
if (cwd / "annotations.db").exists() and cwd.name == "pipeline":
    PROJECT_ROOT = cwd.parent
    DB_PATH = cwd / "annotations.db"
else:
    PROJECT_ROOT = cwd
    DB_PATH = PROJECT_ROOT / "pipeline" / "annotations.db"

DB_DIR = DB_PATH.parent
print("Project root:", PROJECT_ROOT)
print("Database:", DB_PATH)
assert DB_PATH.exists(), f"Database not found: {DB_PATH}"

In [ ]:
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

def query(sql, params=None):
    return pd.read_sql_query(sql, conn, params=params or {})

def table_exists(name):
    row = conn.execute(
        "SELECT 1 FROM sqlite_master WHERE type = 'table' AND name = ?",
        (name,),
    ).fetchone()
    return row is not None

print("Connected")

## Database Summary

In [ ]:
summary = []
for table in ["frames", "annotations", "label_categories", "alpamayo_predictions", "alpamayo_prediction_points"]:
    if table_exists(table):
        count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        summary.append({"table": table, "rows": count})
pd.DataFrame(summary)

## Run Queries

Edit the SQL in these cells to inspect whatever part of the database you need.

In [ ]:
# Example: frames that contain a vehicle label
query("""
SELECT DISTINCT f.id, f.filename, f.source, f.frame_number, f.relative_path
FROM frames f
JOIN annotations a ON a.frame_id = f.id
JOIN label_categories lc ON lc.annotation_id = a.id
WHERE lc.category = 'vehicle' AND lc.present = 1
ORDER BY f.source, f.frame_number
LIMIT 5
""")

## Image Helpers

The DB stores image bytes and paths. These helpers display either:

- an existing preview image from `local_yolo_annotations/preview/`,
- YOLO boxes overlaid from `local_yolo_annotations/labels/`, or
- the stored DB image plus high-level DB labels if no box files are available.

In [ ]:
COLORS = ["#e53935", "#1e88e5", "#43a047", "#fb8c00", "#8e24aa", "#00acc1"]

def get_frame_row(frame_row_or_id):
    if isinstance(frame_row_or_id, int):
        row = conn.execute("SELECT * FROM frames WHERE id=?", (frame_row_or_id,)).fetchone()
    else:
        row = frame_row_or_id
    if row is None:
        raise ValueError("Frame not found")
    return row

def resolve_frame_path(frame_row_or_id):
    row = get_frame_row(frame_row_or_id)
    return (DB_DIR / row["relative_path"]).resolve()

def load_frame_image(frame_row_or_id):
    row = get_frame_row(frame_row_or_id)
    keys = row.keys()
    if "image_data" in keys and row["image_data"] is not None:
        return Image.open(io.BytesIO(row["image_data"])).convert("RGB")
    return Image.open(resolve_frame_path(row)).convert("RGB")

def get_frame_labels(frame_id):
    rows = conn.execute(
        """
        SELECT lc.category, lc.present, lc.confidence
        FROM annotations a
        JOIN label_categories lc ON lc.annotation_id = a.id
        WHERE a.frame_id = ?
        ORDER BY lc.category
        """,
        (frame_id,),
    ).fetchall()
    return [dict(row) for row in rows]

def candidate_annotation_paths(image_path):
    camera_name = image_path.parent.name
    segment_dir = image_path.parent.parent if camera_name.startswith("raw") else image_path.parent
    ann_dir = segment_dir / "local_yolo_annotations"
    camera_preview = ann_dir / camera_name / "preview" / f"{image_path.stem}_annotated.jpg"
    flat_preview = ann_dir / "preview" / f"{image_path.stem}_annotated.jpg"
    camera_label_file = ann_dir / camera_name / "labels" / f"{image_path.stem}.txt"
    flat_label_file = ann_dir / "labels" / f"{image_path.stem}.txt"
    root_classes = ann_dir / "classes.txt"
    camera_classes = ann_dir / camera_name / "classes.txt"
    preview = camera_preview if camera_preview.exists() else flat_preview
    label_file = camera_label_file if camera_label_file.exists() else flat_label_file
    classes_file = root_classes if root_classes.exists() else camera_classes
    return preview, label_file, classes_file

def load_classes(classes_file):
    if not classes_file.exists():
        return []
    return [line.strip() for line in classes_file.read_text().splitlines() if line.strip()]

def draw_yolo_boxes(image_path, label_file, classes_file):
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    classes = load_classes(classes_file)
    w, h = image.size
    if not label_file.exists():
        return image
    for line in label_file.read_text().splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue
        class_id = int(parts[0])
        xc, yc, bw, bh = map(float, parts[1:])
        x1 = (xc - bw / 2) * w
        y1 = (yc - bh / 2) * h
        x2 = (xc + bw / 2) * w
        y2 = (yc + bh / 2) * h
        color = COLORS[class_id % len(COLORS)]
        label = classes[class_id] if class_id < len(classes) else str(class_id)
        draw.rectangle((x1, y1, x2, y2), outline=color, width=3)
        text_box = draw.textbbox((x1, y1), label, font=font)
        draw.rectangle((text_box[0] - 2, text_box[1] - 2, text_box[2] + 4, text_box[3] + 4), fill=color)
        draw.text((x1, y1), label, fill="white", font=font)
    return image

def show_frame(frame_id, prefer_preview=True, figsize=(12, 7)):
    row = conn.execute("SELECT * FROM frames WHERE id=?", (frame_id,)).fetchone()
    if row is None:
        raise ValueError(f"No frame_id={frame_id}")
    image_path = resolve_frame_path(row)
    preview, label_file, classes_file = candidate_annotation_paths(image_path)
    labels = get_frame_labels(frame_id)
    present = [item["category"] for item in labels if item["present"]]

    if prefer_preview and preview.exists():
        image = Image.open(preview).convert("RGB")
        source = f"preview: {preview}"
    elif label_file.exists() and classes_file.exists():
        image = draw_yolo_boxes(image_path, label_file, classes_file)
        source = f"overlay from: {label_file}"
    else:
        image = load_frame_image(row)
        source = "stored DB image; no YOLO box file found"

    plt.figure(figsize=figsize)
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"frame_id={frame_id} | {row['source']} | {row['filename']} | labels={present}\n{source}")
    plt.show()
    return image_path

print("Image helpers loaded")

## Display Annotated Images

In [ ]:
# Pick a frame id from the query output above.
FRAME_ID = query("SELECT id FROM frames ORDER BY source, frame_number LIMIT 1").iloc[0]["id"]
show_frame(int(FRAME_ID))

In [ ]:
# Show a small gallery of frames from a source.
SOURCE = "route_3_segment_00"
frames = query(
    """
    SELECT id, filename, frame_number
    FROM frames
    WHERE source = ?
    ORDER BY frame_number
    LIMIT 6
    """,
    [SOURCE],
)
frames

In [ ]:
for frame_id in frames["id"].tolist():
    show_frame(int(frame_id), figsize=(10, 5))

## Alpamayo Prediction Queries

In [ ]:
if table_exists("alpamayo_predictions"):
    display(query("""
    SELECT
        ap.id AS prediction_id,
        ap.frame_id,
        f.filename,
        f.source,
        f.frame_number,
        ap.nav_command,
        ap.selection_mode,
        ap.selected_sample_index,
        ap.frames_stored,
        ap.created_at
    FROM alpamayo_predictions ap
    JOIN frames f ON f.id = ap.frame_id
    ORDER BY ap.id DESC
    LIMIT 20
    """))
else:
    print("No alpamayo_predictions table yet. Import or generate predictions first.")

## Plot Predictions

This uses the same coordinate convention as the nav notebook/video exporter: forward is `x_m`, lateral is `y_m`, and the plot uses `-y_m` on the horizontal axis.

In [ ]:
def prediction_points(prediction_id):
    return query(
        """
        SELECT step_index, x_m, y_m, z_m
        FROM alpamayo_prediction_points
        WHERE prediction_id = ?
        ORDER BY step_index
        """,
        [prediction_id],
    )

def load_prediction(prediction_id):
    row = conn.execute(
        """
        SELECT ap.*, f.filename, f.source, f.frame_number
        FROM alpamayo_predictions ap
        JOIN frames f ON f.id = ap.frame_id
        WHERE ap.id = ?
        """,
        (prediction_id,),
    ).fetchone()
    if row is None:
        raise ValueError(f"No prediction_id={prediction_id}")
    return dict(row)

def plot_prediction(prediction_id, show_gt=True, show_image=False):
    pred = load_prediction(prediction_id)
    pts = prediction_points(prediction_id)

    if show_image:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        plt.sca(axes[0])
        image_path = resolve_frame_path(pred["frame_id"])
        preview, label_file, classes_file = candidate_annotation_paths(image_path)
        if preview.exists():
            image = Image.open(preview).convert("RGB")
        elif label_file.exists() and classes_file.exists():
            image = draw_yolo_boxes(image_path, label_file, classes_file)
        else:
            image = Image.open(image_path).convert("RGB")
        axes[0].imshow(image)
        axes[0].axis("off")
        ax = axes[1]
    else:
        fig, ax = plt.subplots(figsize=(7, 7))

    ax.plot(-pts["y_m"], pts["x_m"], marker="o", linewidth=2.0, color="lime", label="Selected Prediction")

    x_values = [-float(y) for y in pts["y_m"].tolist()]
    y_values = [float(x) for x in pts["x_m"].tolist()]

    if show_gt and pred.get("gt_path_json"):
        gt = pd.DataFrame(json.loads(pred["gt_path_json"]))
        if not gt.empty:
            ax.plot(-gt["y_m"], gt["x_m"], marker="o", linewidth=2.0, color="red", label="Ground Truth")
            x_values.extend([-float(y) for y in gt["y_m"].tolist()])
            y_values.extend([float(x) for x in gt["x_m"].tolist()])

    ax.plot(0, 0, marker="*", color="black", markersize=14)
    x_values.append(0.0)
    y_values.append(0.0)

    if x_values and y_values:
        min_x, max_x = min(x_values), max(x_values)
        min_y, max_y = min(y_values), max(y_values)
        x_mid = (min_x + max_x) / 2.0
        y_mid = (min_y + max_y) / 2.0
        x_half = max((max_x - min_x) / 2.0, 5.0)
        y_half = max((max_y - min_y) / 2.0, 5.0)
        ax.set_xlim(x_mid - x_half, x_mid + x_half)
        ax.set_ylim(y_mid - y_half, y_mid + y_half)

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("-lateral y (m)")
    ax.set_ylabel("forward x (m)")
    ax.grid(True, alpha=0.25)
    ax.legend()
    ax.set_title(
        f"prediction_id={prediction_id} | frame_id={pred['frame_id']} | {pred['source']} #{pred['frame_number']}\n"
        f"nav={pred['nav_command']}"
    )
    plt.show()
    if pred.get("cot"):
        print(pred["cot"])

def latest_prediction_id():
    row = conn.execute("SELECT id FROM alpamayo_predictions ORDER BY id DESC LIMIT 1").fetchone()
    return row[0] if row else None

print("Prediction plotting helpers loaded")

In [ ]:
PREDICTION_ID = latest_prediction_id()
print("Latest prediction:", PREDICTION_ID)
if PREDICTION_ID is not None:
    plot_prediction(PREDICTION_ID, show_gt=True, show_image=True)

## Export Query Results

Use this pattern to save any SQL result to CSV.

In [ ]:
df = query("""
SELECT
    ap.id AS prediction_id,
    ap.frame_id,
    f.filename,
    f.source,
    f.frame_number,
    ap.nav_command,
    ap.frames_stored,
    ap.created_at
FROM alpamayo_predictions ap
JOIN frames f ON f.id = ap.frame_id
ORDER BY ap.id
""") if table_exists("alpamayo_predictions") else pd.DataFrame()

out_csv = PROJECT_ROOT / "alpamayo_prediction_summary.csv"
df.to_csv(out_csv, index=False)
print(f"Wrote {len(df)} rows to {out_csv}")
df.head()